# Avaliando LLMs com LangChain

Avaliar LLMs é um desafio: benchmarks tradicionais nem sempre funcionam para modelos generativos, já que uma resposta correta pode ser formulada de várias maneiras diferentes.

Duas abordagens se destacam: a **avaliação humana** (precisa, mas cara e lenta para escalar) e o **LLM-as-a-judge** (usar um LLM como juiz — pesquisas recentes mostram mais de 80% de concordância com as preferências humanas).

Neste notebook, veremos na prática três formas de avaliar LLMs com LangChain:

- **Avaliação por critérios**: concisão, correção e critérios personalizados
- **Avaliação de RAG**: se o modelo usa corretamente o contexto fornecido para responder
- **Comparação par a par e pontuação**: útil para gerar feedback de IA para RLAIF ou DPO

**Modelos utilizados (via Hugging Face Inference API):**

- **Modelo avaliado**: [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) (variável `HF_MODEL_ID`)
- **Modelo juiz (avaliador)**: [Qwen/Qwen3-14B](https://huggingface.co/Qwen/Qwen3-14B) (variável `HF_MODEL_EVAL_ID`)

É possível usar qualquer LLM suportado pelo LangChain como avaliador. As credenciais são carregadas do arquivo `.env` na raiz do projeto.

In [62]:
%pip install --upgrade huggingface_hub langchain langchain_huggingface langchain-classic langchain-core langchain-community langchain-openai transformers dotenv torch --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


*Nota: é preciso ter uma conta no [huggingface.co](https://huggingface.co) com acesso aos modelos Qwen via Inference API.*

In [63]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env na raiz do projeto
load_dotenv(dotenv_path=Path.cwd() / ".env")

# Exemplo: acessar variáveis com os.getenv(...)
# os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
# ou simplesmente:
# hf_token = os.getenv("HF_TOKEN")

True

In [64]:
import os

# Mostra as variáveis relevantes sem expor os valores secretos
for key in ["HF_TOKEN", "OPENAI_API_KEY", "HF_MODEL_ID"]:
    value = os.getenv(key)
    if value:
        masked = f"{value[:4]}...{value[-4:]}" if len(value) > 8 else "***"
        print(f"{key}: SET ({masked})")
    else:
        print(f"{key}: MISSING")

'HF_TOKEN: SET (hf_B...GLEa)'
'OPENAI_API_KEY: SET (sk-p...xo4A)'
'HF_MODEL_ID: SET (Qwen...3-8B)'


In [65]:
import os

from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv()

HF_TOKEN = os.environ["HF_TOKEN"]

login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [66]:
hf_model_id = os.getenv("HF_MODEL_ID")
hf_model_id

'Qwen/Qwen3-8B'

In [67]:
from transformers import AutoTokenizer
from huggingface_hub import InferenceClient


HF_TOKEN = os.environ["HF_TOKEN"]
HF_MODEL_ID = os.environ["HF_MODEL_ID"]

# Cria o cliente da Hugging Face
hf_client = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto",
)

def generate(text):
    response = hf_client.chat.completions.create(
        model=HF_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": text,
            }
        ],
        temperature=0.6,
        max_tokens=2048,
        top_p = 0.9
    )

    return response.choices[0].message.content.strip()

## Avaliação por critérios

A avaliação por critérios mede atributos específicos da geração em vez de depender de uma métrica única, gerando pontuações interpretáveis. Vamos avaliar a resposta do prompt abaixo quanto a:

- **concisão** da geração, ou seja o quão concisa é a reposta
- **correção** usando uma referência adicional
- **critério personalizado**: se a resposta é explicada para uma criança de 5 anos

In [68]:
prompt = "Quem é o atual presidente do Brasil?"

Vamos primeiro ver o que o modelo gera para o prompt:

In [69]:
pred = generate(prompt)
print(pred)

('O atual presidente do Brasil é **Jair Messias Bolsonaro**, eleito em 2018 e '
 'reeleito em 2022. Ele está no cargo desde 1º de janeiro de 2019, com mandato '
 'até 31 de dezembro de 2026.')


O avaliador de critérios retorna um dicionário com os seguintes valores:

- `score`: inteiro binário (0 ou 1), onde 1 significa que a resposta atende ao critério
- `value`: "Y" ou "N", correspondente ao score
- `reasoning`: raciocínio em cadeia gerado pelo LLM juiz antes da pontuação

Para saber mais, consulte a [documentação](https://python.langchain.com/docs/guides/evaluation/string/criteria_eval_chain).

### Avaliação de concisão

Concisão mede se a resposta é direta e objetiva.

In [70]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.evaluation import load_evaluator

HF_MODEL_EVAL_ID = os.environ["HF_MODEL_EVAL_ID"]


# Modelo juiz: Qwen/Qwen3-14B via Hugging Face
evaluation_hf = HuggingFaceEndpoint(
    repo_id=HF_MODEL_EVAL_ID,
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.6,
    max_tokens=1024,
    top_p = 0.9
)

# Empacota o LLM da HF como um chat model do LangChain
evaluation_llm = ChatHuggingFace(
    llm=evaluation_hf,
)

# Cria o avaliador
evaluator = load_evaluator(
    "criteria",
    criteria="conciseness",
    llm=evaluation_llm,
)

# Avalia
eval_result = evaluator.evaluate_strings(
    prediction=pred,
    input=prompt,
)

print(eval_result)

WARNING! max_tokens is not default parameter.
                    max_tokens was transferred to model_kwargs.
                    Please make sure that max_tokens is what you intended.


{'reasoning': 'N', 'score': 0, 'value': 'N'}


Concordamos com o raciocínio do Qwen3-14B: a resposta mais concisa seria apenas o nome do presidente.

### Correção usando uma referência adicional

Avaliar a correção apenas com o conhecimento interno do LLM juiz pode ser arriscado, pois não temos garantia de que ele sabe a resposta certa. Por isso, criamos o avaliador com `requires_reference=True`, usando uma referência externa.

Como referência, usamos o texto: _"O novo e 39º presidente do Brasil é Philipp Schmid."_ — obviamente errado, para testar se o juiz prioriza a referência em vez do próprio conhecimento.

In [71]:
from langchain_classic.evaluation import load_evaluator
from pprint import pprint as print

# cria o avaliador
evaluator = load_evaluator("labeled_criteria", criteria="correctness", llm=evaluation_llm,requires_reference=True)

# avalia
eval_result = evaluator.evaluate_strings(
    prediction=pred,
    input=prompt,
    reference="O novo e 39º presidente do Brasil é Philipp Schmid."
)

# exibe o resultado
print(eval_result)

{'reasoning': 'N', 'score': 0, 'value': 'N'}


Resultado interessante: o juiz priorizou o conteúdo contido no reference no lugar de seu conhecimento interno.

### Critério personalizado: explicação para uma criança de 5 anos

O LangChain permite definir critérios personalizados. Neste exemplo, avaliamos se a geração está explicada de forma que uma criança de 5 anos entenderia:

In [74]:
from langchain_classic.evaluation import load_evaluator
from pprint import pprint as print

# critério personalizado
custom_criterion = {"teste": "A resposta está explicada de forma que uma criança entenderia?"}

# cria o avaliador
evaluator = load_evaluator("criteria", criteria=custom_criterion, llm=evaluation_llm)

# avalia
eval_result = evaluator.evaluate_strings(
    prediction=pred,
    input=prompt,
)

# exibe o resultado
print(eval_result)

{'reasoning': 'Y', 'score': 1, 'value': 'Y'}


O raciocínio do Qwen3-14B faz sentido: uma criança provavelmente não entenderia termos como "mandato". Como este é apenas um exemplo de critério personalizado, seguimos em frente.

## Avaliação de RAG (Retrieval Augmented Generation)

RAG é um dos casos de uso mais populares de LLMs — e um dos mais difíceis de avaliar: queremos saber se o modelo usou corretamente o contexto fornecido para responder.

O LangChain oferece o avaliador `context_qa`, que recebe um `context` e uma `question`, além da `prediction` e de uma `reference`, para avaliar a correção da geração. Ele retorna um dicionário com:

- `reasoning`: raciocínio do LLM juiz antes da nota
- `score`: 1 (correto) ou 0 (incorreto)
- `value`: "CORRECT" ou "INCORRECT"

In [75]:
question = "Quantas pessoas moram em Belo Horizonte?"
context="Belo Horizonte é a capital do estado de Minas Gerais. Com 2.315.560 habitantes, segundo o Censo 2022 do IBGE, é a sexta cidade mais populosa do Brasil. Fundada em 1897 para ser a nova capital mineira, a cidade é conhecida pelo conjunto arquitetônico da Pampulha, projetado por Oscar Niemeyer, e por sua tradicional cultura de botecos."

prompt = f"""Use os trechos de contexto a seguir para responder à pergunta no final. Se você não souber a resposta, diga apenas que não sabe, não tente inventar uma resposta.

{context}

Pergunta: {question}"""

pred = generate(prompt)
print(pred)

'Segundo o Censo 2022 do IBGE, Belo Horizonte tem 2.315.560 habitantes.'


Podemos também testar rapidamente como o Qwen3-8B responde sem o contexto:

In [76]:
false_pred = generate(question)
print(false_pred)

('A população estimada de Belo Horizonte, capital do estado de Minas Gerais, '
 'Brasil, é de aproximadamente **2,5 milhões de habitantes** (dados de 2020, '
 'segundo o Instituto Brasileiro de Geografia e Estatística - IBGE). \n'
 '\n'
 'É importante ressaltar que essa estimativa pode variar conforme fontes e '
 'métodos de cálculo, e a população pode ter crescido ligeiramente nos últimos '
 'anos devido a fatores como migração, natalidade e outros. Para dados mais '
 'atualizados, recomenda-se consultar o IBGE ou outras fontes oficiais. \n'
 '\n'
 'Se quiser, posso te ajudar a encontrar a população da região metropolitana '
 'ou de outras áreas específicas! 🌆')


Sem o contexto, a geração tende a ficar incorreta. Vamos ver se o avaliador detecta isso. Como referência, usamos o número bruto: `2.315.560`.

In [77]:
from langchain_classic.evaluation import load_evaluator
from pprint import pprint as print

# cria o avaliador
evaluator = load_evaluator("context_qa", llm=evaluation_llm)

# avalia
eval_result = evaluator.evaluate_strings(
  input=question,
  prediction=pred,
  context=context,
  reference="2.315.560"
)

# exibe o resultado
print(eval_result)

{'reasoning': 'GRADE: CORRECT', 'score': 1, 'value': 'CORRECT'}


O avaliador classificou a geração como correta. Agora vamos testar o que acontece com a resposta gerada sem contexto:

In [81]:
# avalia
eval_result = evaluator.evaluate_strings(
  input=question,
  prediction=false_pred,
  context=context,
  reference="2.315.560"
)

# exibe o resultado
print(eval_result)

{'reasoning': 'N', 'score': 0, 'value': 'N'}


O avaliador detectou que a geração está incorreta.

Alternativamente, se você não tiver uma referência, pode reutilizar o avaliador de critérios (`labeled_criteria`) para avaliar a correção, usando a pergunta como `input` e o contexto como `reference`:

In [82]:
from langchain_classic.evaluation import load_evaluator
from pprint import pprint as print

# cria o avaliador
evaluator = load_evaluator("labeled_criteria", criteria="correctness", llm=evaluation_llm, requires_reference=True)

# avalia
eval_result = evaluator.evaluate_strings(
    prediction=pred,
    input=question,
    reference=context,
)

# exibe o resultado
print(eval_result)

{'reasoning': 'Y', 'score': 1, 'value': 'Y'}


Como vemos, o Qwen3-14B concluiu corretamente que a geração está de acordo com o contexto fornecido.

## Comparação par a par e pontuação

Na comparação par a par, o juiz escolhe a melhor entre duas gerações; na pontuação, ele atribui notas à qualidade de cada uma. Esses métodos são úteis para comparar modelos ou versões e para gerar dados de preferência (feedback de IA) para RLAIF ou DPO.

RLAIF:  Substitui ou complementa o feedback humano por notas ou escolhas feitas por um modelo de IA mais forte (como o GPT-4). O objetivo é baratear e acelerar a criação de dados de preferência. [1] (https://www.datacamp.com/pt/blog/rlaif-reinforcement-learning-from-ai-feedback), [2] (https://medium.com/foundation-models-deep-dive/beyond-traditional-rlhf-exploring-dpo-constitutional-ai-and-the-future-of-llm-alignment-bc30089644c9)

DPO (Direct Preference Optimization): É um método de treinamento que elimina a necessidade de treinar um modelo de recompensa separado e de usar aprendizado por reforço clássico (como PPO). Ele otimiza o modelo diretamente usando pares de respostas preferidas/rejeitadas. [1] (https://www.youtube.com/watch?v=x5eWzxzOImY&t=184), [2] (https://medium.com/@baicenxiao/rlhf-vs-dpo-choosing-the-method-for-llm-alignment-tuning-66f45ef3d4b5)

Primeiro, geramos duas respostas para o mesmo prompt e depois pedimos ao juiz que escolha entre elas.

In [83]:
prompt = "Por favor, escreva um e-mail curto para o seu chefe sobre a reunião de amanhã."
pred_a = generate(prompt)

prompt = "Escreva um e-mail para o seu chefe sobre a reunião de amanhã" # sem o ponto final, para não usar resultado em cache
pred_b = generate(prompt)

assert pred_a != pred_b

Agora vamos pedir ao juiz (Qwen3-14B) que escolha a geração preferida:

In [84]:
from langchain_core.prompts import ChatPromptTemplate

judge_prompt = ChatPromptTemplate.from_template("""
Compare as duas respostas para a pergunta abaixo.

Pergunta:
{input}

Resposta A:
{prediction}

Resposta B:
{prediction_b}

Escolha a melhor resposta:

A = Resposta A é melhor
B = Resposta B é melhor
C = As duas são equivalentes

Responda SOMENTE:
[[A]]
[[B]]
ou
[[C]]
""")

judge_chain = judge_prompt | evaluation_llm

result = judge_chain.invoke({
    "input": prompt,
    "prediction": pred_a,
    "prediction_b": pred_b,
})

verdict = result.content.strip()

print(repr(verdict))

"'[[B]]'"


Com a escolha do juiz, poderíamos gerar feedback de IA para RLAIF ou DPO. A seguir, pontuamos cada geração individualmente para uma avaliação mais detalhada.

In [85]:
from langchain_classic.evaluation import load_evaluator
from pprint import pprint as print

# cria o avaliador
evaluator = load_evaluator("score_string", llm=evaluation_llm)

# avalia
eval_result_a = evaluator.evaluate_strings(
    prediction=pred_a,
    input=prompt,
)
eval_result_b = evaluator.evaluate_strings(
    prediction=pred_b,
    input=prompt,
)


# exibe o resultado
print(f"Score A: {eval_result_a['score']}")
print(f"Score B: {eval_result_b['score']}")

This chain was only tested with GPT-4. Performance may be significantly worse with other models.


'Score A: 9'
'Score B: 9'
